# ADAPT-VQE with native FlagQuantum synthesis

Use the original four-qubit Hamiltonian, all 56 ordered pool entries and exact statevector expectations. Compute all gradients as expectations of the source commutators. Select the lowest signed derivative, keep repeated selections, and apply the source stopping rule with tolerance 0.001 and at most ten adaptive steps. The local optimizer uses Nelder–Mead with a 100-iteration budget and a simplex offset of 0.2. It reevaluates the synthesized quantum state at every objective query.

In the source standard loop, the optimization circuit captures the previous declared depth before the newest selected operator is active. This matches the stored first energy of the bare reference. The modified loop begins at pool index 10 and fits its active operators before screening. The source fixed wrapper supplies a modular schedule, while its helper reads the modified loop's shared selection list. The Notebook evaluates the modular schedule and the captured schedule separately. Local execution retains the actual resulting trajectories and reports both static width and actual depth.

Run with FlagQuantum, Matplotlib and Jupyter installed.


In [ ]:
from importlib import import_module
from pathlib import Path

from examples.synthesis.algorithms.quantum_state_preparation.adapt_vqe.overview.main import (
    save_run,
    show_fixed,
    show_run,
)
from examples.synthesis.algorithms.quantum_state_preparation.adapt_vqe.overview.model import (
    HAMILTONIAN_DATA,
    INNER_BUDGET,
    OUTER_BUDGET,
    POOL,
    TOLERANCE,
    build_model,
)
from examples.synthesis.algorithms.quantum_state_preparation.adapt_vqe.overview.quantum import (
    build_program,
)
from examples.synthesis.algorithms.quantum_state_preparation.adapt_vqe.overview.workflow import (
    adapt,
    fixed,
    screen,
)


def show(path: Path) -> None:
    viewer = import_module("IPython.display")
    viewer.display(viewer.Image(filename=str(path)))


## Hamiltonian and full gradient pool

In [ ]:
model = build_model()
print("Screening operator shape:", model.screening.shape)


In [ ]:
print("Hamiltonian terms:", len(HAMILTONIAN_DATA), "Pool entries:", len(POOL))


In [ ]:
print(f"Exact ground energy: {model.ground_energy:.12f}")


## Native variable-length ansatz

In [ ]:
initial_program = build_program((), ())
print("Native reference program:", initial_program.report["resources"])


In [ ]:
initial_gradients = screen(model, (), ())
print("Source limits and tolerance:", OUTER_BUDGET, INNER_BUDGET, TOLERANCE)
print("Initial gradient minimum:", min(initial_gradients))


## Standard delayed-growth loop

In [ ]:
standard = adapt(model, modified=False)
show_run("Standard", standard, model.ground_energy)


In [ ]:
print("Best standard active operators:", standard.best.fit.indices)


In [ ]:
standard_energy, standard_gradient = save_run("Standard", standard, model.ground_energy)
show(standard_energy)


In [ ]:
show(standard_gradient)


In [ ]:
print(
    "Actual standard width and depth:",
    standard.best.fit.resources.n_qubits,
    standard.best.fit.resources.depth,
)


## Modified ADAPT-VQE

In [ ]:
modified = adapt(model, modified=True)
show_run("Modified", modified, model.ground_energy)


In [ ]:
print("Best modified active operators:", modified.best.fit.indices)


In [ ]:
modified_energy, modified_gradient = save_run("Modified", modified, model.ground_energy)
show(modified_energy)


In [ ]:
show(modified_gradient)


In [ ]:
print(
    "Actual modified width and depth:",
    modified.best.fit.resources.n_qubits,
    modified.best.fit.resources.depth,
)


## Fixed comparisons

In [ ]:
modular = fixed(model)
show_fixed("Modular fixed", modular, model.ground_energy)
if len(modified.selections) == OUTER_BUDGET:
    captured = fixed(model, modified.selections)
    show_fixed("Captured fixed", captured, model.ground_energy)


In [ ]:
print(
    "Best modular width and depth:", modular.resources.n_qubits, modular.resources.depth
)
